# 13. リターン分布 & 市場レジーム EDA

**目的**: 日本株25銘柄のリターン分布・自己相関・セクター特性・市場レジームを探索的に分析し、予測モデル構築の前提を整理する。

| 項目 | 内容 |
|------|------|
| データ | `equity_jp_ohlcv.parquet`, `macro.parquet` |
| 銘柄数 | 25銘柄（東証プライム主要株） |
| 分析期間 | データ全期間 |

---
## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
import yaml

# scipy stats
from scipy.stats import skew, kurtosis, jarque_bera, norm
import scipy.stats as stats

# matplotlib / seaborn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# statsmodels
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox

# japanize
try:
    import japanize_matplotlib
    print('japanize_matplotlib loaded')
except ImportError:
    plt.rcParams['font.family'] = ['DejaVu Sans', 'Arial Unicode MS', 'Hiragino Sans']
    print('japanize_matplotlib not found — using fallback font')

%matplotlib inline

sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (12, 5)})

print('Setup complete')

---
## 1. データ読み込み

In [ ]:
DATA_DIR = Path('../../data/')
CONFIG_PATH = Path('../../configs/stock_config.yaml')

SYMBOLS = [
    '7203.T', '6758.T', '8306.T', '9432.T', '9984.T',
    '6861.T', '8316.T', '4063.T', '6367.T', '7974.T',
    '8035.T', '6954.T', '4519.T', '9433.T', '3382.T',
    '4502.T', '7751.T', '6098.T', '8766.T', '6501.T',
    '4568.T', '7267.T', '6702.T', '6503.T', '5108.T'
]

SECTOR_MAP = {
    '7203.T': 'Manuf',   '7267.T': 'Manuf',   '7751.T': 'Manuf',
    '6758.T': 'Tech',    '7974.T': 'Tech',     '6861.T': 'Tech',
    '8306.T': 'Finance', '8316.T': 'Finance',  '8766.T': 'Finance',
    '4568.T': 'Pharma',  '4519.T': 'Pharma',   '4502.T': 'Pharma',
    '9432.T': 'Service', '9984.T': 'Service',  '3382.T': 'Service',
    '9433.T': 'Service', '6098.T': 'Service',
    '4063.T': 'Industrial', '6367.T': 'Industrial', '8035.T': 'Industrial',
    '6954.T': 'Industrial', '6501.T': 'Industrial', '6702.T': 'Industrial',
    '6503.T': 'Industrial', '5108.T': 'Industrial',
}

SYMBOL_NAME = {
    '7203.T': 'Toyota',      '6758.T': 'Sony',        '8306.T': 'MUFG',
    '9432.T': 'NTT',         '9984.T': 'SoftBank',    '6861.T': 'Keyence',
    '8316.T': 'SMFG',        '4063.T': 'Shin-Etsu',   '6367.T': 'Daikin',
    '7974.T': 'Nintendo',    '8035.T': 'TokyoElec',   '6954.T': 'Fanuc',
    '4519.T': 'Chugai',      '9433.T': 'KDDI',        '3382.T': 'Seven&i',
    '4502.T': 'Takeda',      '7751.T': 'Canon',       '6098.T': 'Recruit',
    '8766.T': 'TokioMarine', '6501.T': 'Hitachi',     '4568.T': 'DaiichiSankyo',
    '7267.T': 'Honda',       '6702.T': 'Fujitsu',     '6503.T': 'MitsubishiElec',
    '5108.T': 'Bridgestone',
}

# ---- Load OHLCV ----
ohlcv_path = DATA_DIR / 'equity_jp_ohlcv.parquet'
macro_path = DATA_DIR / 'macro.parquet'

try:
    df = pd.read_parquet(ohlcv_path)
    macro = pd.read_parquet(macro_path)
    print('Real data loaded.')
except Exception as e:
    print(f'Parquet not found ({e}). Generating dummy data...')

    rng = np.random.default_rng(42)
    n_days = 1000
    dates = pd.bdate_range('2020-01-01', periods=n_days, freq='B')

    records = []
    for sym in SYMBOLS:
        price = 1000.0
        for d in dates:
            ret = rng.normal(0.0003, 0.018)
            price = max(price * np.exp(ret), 1.0)
            hi = price * (1 + abs(rng.normal(0, 0.005)))
            lo = price * (1 - abs(rng.normal(0, 0.005)))
            vol = rng.integers(100_000, 5_000_000)
            records.append({'date': d, 'symbol': sym,
                            'open': price, 'high': hi, 'low': lo,
                            'close': price, 'volume': float(vol)})

    df = pd.DataFrame(records)

    macro_ret = rng.normal(0.0003, 0.012, n_days)
    nk = np.cumprod(np.exp(macro_ret)) * 22000
    dj = np.cumprod(np.exp(rng.normal(0.0004, 0.010, n_days))) * 30000
    macro = pd.DataFrame({
        'date': dates,
        'nikkei225': nk,
        'dow_jones': dj,
        'usd_index': 100 + np.cumsum(rng.normal(0, 0.2, n_days)),
        'crude_oil': 60 + np.cumsum(rng.normal(0, 0.3, n_days)),
        'gold': 1800 + np.cumsum(rng.normal(0, 5, n_days)),
        'us10y_yield': 2.0 + np.cumsum(rng.normal(0, 0.02, n_days)),
        'usdjpy': 110 + np.cumsum(rng.normal(0, 0.3, n_days)),
    })
    print('Dummy data created.')

# ---- Normalise dtypes ----
df['date'] = pd.to_datetime(df['date'])
macro['date'] = pd.to_datetime(macro['date'])

print(f'\nOHLCV shape : {df.shape}')
print(f'Date range  : {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'Symbols     : {df["symbol"].nunique()}')
print(f'\nMacro shape : {macro.shape}')
print(f'Macro cols  : {list(macro.columns)}')
df.head(3)

---
## 2. リターン計算

- **対数リターン** (log return): `log(P_t / P_{t-1})`
- **単純リターン** (simple return): `(P_t - P_{t-1}) / P_{t-1}`
- 1日・5日・20日のホライズンを計算

In [ ]:
df = df.sort_values(['symbol', 'date']).reset_index(drop=True)

for h, label in [(1, '1d'), (5, '5d'), (20, '20d')]:
    log_col  = f'logret_{label}'
    simp_col = f'ret_{label}'
    df[log_col]  = df.groupby('symbol')['close'].transform(
        lambda x: np.log(x / x.shift(h))
    )
    df[simp_col] = df.groupby('symbol')['close'].transform(
        lambda x: x.pct_change(h)
    )

print('Return columns added:')
ret_cols = [c for c in df.columns if c.startswith('logret_') or c.startswith('ret_')]
print(ret_cols)
print(f'\nTotal rows: {len(df):,}  |  NaN logret_1d: {df["logret_1d"].isna().sum():,}')
df[['date','symbol','close','logret_1d','ret_1d','logret_5d','logret_20d']].dropna().head(6)

---
## 3. リターン分布分析

### 3a. ヒストグラム + KDE (全銘柄プール)

In [ ]:
pooled = df['logret_1d'].dropna()
mu, sigma = pooled.mean(), pooled.std()

fig, ax = plt.subplots(figsize=(12, 5))
sns.histplot(pooled, bins=120, stat='density', kde=True, color='steelblue',
             alpha=0.5, label='実データ KDE', ax=ax)

# Normal overlay
x = np.linspace(pooled.quantile(0.001), pooled.quantile(0.999), 300)
ax.plot(x, norm.pdf(x, mu, sigma), 'r-', lw=2, label=f'正規分布 N({mu:.4f}, {sigma:.4f}²)')

ax.set_title('日次対数リターン 分布 (全銘柄プール)', fontsize=14)
ax.set_xlabel('対数リターン')
ax.set_ylabel('密度')
ax.legend()
plt.tight_layout()
plt.show()

### 3b. QQ プロット (正規分布との比較)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
(osm, osr), (slope, intercept, r) = stats.probplot(pooled, dist='norm')
ax.plot(osm, osr, '.', alpha=0.3, ms=3, label='実データ')
ax.plot(osm, slope * np.array(osm) + intercept, 'r-', lw=2, label='正規理論線')
ax.set_title('QQ プロット — 日次対数リターン vs 正規分布', fontsize=13)
ax.set_xlabel('理論分位数')
ax.set_ylabel('実測分位数')
ax.legend()
plt.tight_layout()
plt.show()

### 3c. 銘柄別 歪度・尖度 テーブル

In [ ]:
stats_rows = []
for sym, grp in df.groupby('symbol')['logret_1d']:
    s = grp.dropna()
    if len(s) < 10:
        continue
    stats_rows.append({
        'symbol': sym,
        'name': SYMBOL_NAME.get(sym, sym),
        'sector': SECTOR_MAP.get(sym, 'Other'),
        'count': len(s),
        'mean': s.mean(),
        'std': s.std(),
        'skewness': skew(s),
        'kurtosis': kurtosis(s),  # excess kurtosis
    })

stats_df = pd.DataFrame(stats_rows).sort_values('kurtosis', ascending=False).reset_index(drop=True)
stats_df['mean_%'] = stats_df['mean'] * 100
stats_df['std_%']  = stats_df['std']  * 100

display_cols = ['symbol','name','sector','count','mean_%','std_%','skewness','kurtosis']
print('銘柄別リターン統計 (超過尖度 降順):')
stats_df[display_cols].style.format({
    'mean_%': '{:.4f}', 'std_%': '{:.4f}',
    'skewness': '{:.3f}', 'kurtosis': '{:.3f}'
}).background_gradient(subset=['kurtosis'], cmap='YlOrRd')

### 3d. ファットテール検定 — Jarque-Bera

In [ ]:
jb_rows = []
for sym, grp in df.groupby('symbol')['logret_1d']:
    s = grp.dropna()
    if len(s) < 10:
        continue
    stat, p = jarque_bera(s)
    jb_rows.append({'symbol': sym, 'name': SYMBOL_NAME.get(sym, sym),
                    'JB_stat': stat, 'p_value': p,
                    'reject_normal': p < 0.05})

jb_df = pd.DataFrame(jb_rows).sort_values('p_value').reset_index(drop=True)
n_reject = jb_df['reject_normal'].sum()
print(f'Jarque-Bera 検定 (有意水準 5%): {n_reject}/{len(jb_df)} 銘柄で正規性を棄却')
jb_df.style.format({'JB_stat': '{:.1f}', 'p_value': '{:.4e}'}) \
     .applymap(lambda v: 'background-color: salmon' if v else '',
               subset=['reject_normal'])

---
## 4. 自己相関分析

### 4a. ACF / PACF — クロスセクション平均リターン

In [ ]:
avg_ret = df.dropna(subset=['logret_1d']).groupby('date')['logret_1d'].mean()
avg_ret = avg_ret.sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(avg_ret, lags=20, ax=axes[0], title='ACF — クロスセクション平均日次リターン')
plot_pacf(avg_ret, lags=20, ax=axes[1], title='PACF — クロスセクション平均日次リターン', method='ywm')
plt.tight_layout()
plt.show()

### 4b. ラグ別自己相関 (Pearson / Spearman) — プール日次リターン

In [ ]:
lags = range(1, 21)
pearson_corrs, spearman_corrs = [], []

# Build per-symbol lag-shifted series then pool
lag_frames = []
for sym, grp in df.groupby('symbol'):
    g = grp[['date', 'logret_1d']].dropna().set_index('date').sort_index()
    for lag in lags:
        tmp = pd.DataFrame({'ret': g['logret_1d'], 'lagged': g['logret_1d'].shift(lag), 'lag': lag})
        lag_frames.append(tmp.dropna())

lag_all = pd.concat(lag_frames)

for lag in lags:
    sub = lag_all[lag_all['lag'] == lag]
    pearson_corrs.append(sub['ret'].corr(sub['lagged'], method='pearson'))
    spearman_corrs.append(sub['ret'].corr(sub['lagged'], method='spearman'))

fig, ax = plt.subplots(figsize=(12, 4))
x = np.array(list(lags))
w = 0.35
ax.bar(x - w/2, pearson_corrs,  width=w, label='Pearson',  alpha=0.8)
ax.bar(x + w/2, spearman_corrs, width=w, label='Spearman', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('ラグ (日)')
ax.set_ylabel('自己相関係数')
ax.set_title('リターン自己相関 — ラグ 1〜20日 (全銘柄プール)')
ax.legend()
plt.tight_layout()
plt.show()

### 4c. 二乗リターン自己相関 (ボラティリティクラスタリング)

In [ ]:
sq_frames = []
for sym, grp in df.groupby('symbol'):
    g = grp[['date', 'logret_1d']].dropna().set_index('date').sort_index()
    g['sq'] = g['logret_1d'] ** 2
    for lag in lags:
        tmp = pd.DataFrame({'sq': g['sq'], 'lagged_sq': g['sq'].shift(lag), 'lag': lag})
        sq_frames.append(tmp.dropna())

sq_all = pd.concat(sq_frames)
sq_corrs = [sq_all[sq_all['lag']==lag]['sq'].corr(sq_all[sq_all['lag']==lag]['lagged_sq'])
            for lag in lags]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(list(lags), sq_corrs, color='coral', alpha=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.set_xlabel('ラグ (日)')
ax.set_ylabel('自己相関係数')
ax.set_title('二乗リターン自己相関 (ボラティリティクラスタリング) — ラグ 1〜20日')
plt.tight_layout()
plt.show()

### 4d. Ljung-Box 検定

In [ ]:
lb_result = acorr_ljungbox(avg_ret.dropna(), lags=[1, 5, 10, 20], return_df=True)
lb_result.index.name = 'lag'
lb_result.columns = ['LB_stat', 'p_value']
lb_result['reject_H0 (5%)'] = lb_result['p_value'] < 0.05
print('Ljung-Box 検定 — クロスセクション平均リターン:')
lb_result.style.format({'LB_stat': '{:.3f}', 'p_value': '{:.4f}'}) \
         .applymap(lambda v: 'background-color: salmon' if v else '',
                   subset=['reject_H0 (5%)'])

---
## 5. クロスセクション統計

### 5a–b. 月次クロスセクション分散度 & 平均リターン

In [ ]:
daily_cs = df.dropna(subset=['logret_1d']).groupby('date')['logret_1d'].agg(
    cs_mean='mean', cs_std='std'
).reset_index().set_index('date').sort_index()

# Monthly resample
monthly_disp = daily_cs['cs_std'].resample('ME').mean()
monthly_mean = daily_cs['cs_mean'].resample('ME').mean()

print(f'月次クロスセクション分散度 stats:\n{monthly_disp.describe().round(6)}')

### 5c. 分散度 vs Nikkei225 月次リターン (ツイン軸)

In [ ]:
# Nikkei monthly return
nk = macro.set_index('date')['nikkei225'].sort_index()
nk_monthly = np.log(nk).resample('ME').last().diff().rename('nk_ret')

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.fill_between(monthly_disp.index, monthly_disp, alpha=0.5, color='steelblue', label='分散度 (CS std)')
ax1.set_xlabel('年月')
ax1.set_ylabel('クロスセクション分散度', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(nk_monthly.index, nk_monthly, color='tomato', lw=1.5, label='Nikkei225 月次対数リターン')
ax2.set_ylabel('Nikkei225 月次リターン', color='tomato')
ax2.tick_params(axis='y', labelcolor='tomato')
ax2.axhline(0, color='tomato', lw=0.5, ls='--')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
ax1.set_title('月次クロスセクション分散度 vs Nikkei225 月次リターン', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. セクター分析

### 6a. セクター列追加

In [ ]:
df['sector'] = df['symbol'].map(SECTOR_MAP).fillna('Other')
df['name']   = df['symbol'].map(SYMBOL_NAME).fillna(df['symbol'])
print('セクター分布:')
print(df[['symbol','sector']].drop_duplicates().groupby('sector').size().sort_values(ascending=False))

### 6b. セクター別リターン Boxplot

In [ ]:
sector_order = ['Manuf', 'Tech', 'Finance', 'Pharma', 'Service', 'Industrial']
plot_data = df.dropna(subset=['logret_1d'])

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=plot_data, x='sector', y='logret_1d', hue='sector',
            order=sector_order, palette='tab10', showfliers=False,
            legend=False, ax=ax)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('セクター別 日次対数リターン分布 (外れ値除外)', fontsize=13)
ax.set_xlabel('セクター')
ax.set_ylabel('日次対数リターン')
plt.tight_layout()
plt.show()

### 6c. セクター別 Beta vs Nikkei225

In [ ]:
# Nikkei daily log return
nk_daily = np.log(macro.set_index('date')['nikkei225'].sort_index()).diff().rename('nk_logret')

beta_rows = []
for sym, grp in df.groupby('symbol'):
    g = grp[['date','logret_1d','sector']].dropna().set_index('date')
    merged = g.join(nk_daily, how='inner').dropna()
    if len(merged) < 20:
        continue
    cov_val = np.cov(merged['logret_1d'], merged['nk_logret'])[0, 1]
    var_val = merged['nk_logret'].var()
    beta = cov_val / var_val if var_val > 0 else np.nan
    beta_rows.append({'symbol': sym, 'name': SYMBOL_NAME.get(sym, sym),
                      'sector': SECTOR_MAP.get(sym, 'Other'), 'beta': beta})

beta_df = pd.DataFrame(beta_rows)
sector_beta = beta_df.groupby('sector')['beta'].agg(['mean','std','count']).round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart sector average beta
sector_beta['mean'].reindex(sector_order).plot(
    kind='bar', ax=axes[0], color='steelblue', alpha=0.8, yerr=sector_beta['std'].reindex(sector_order))
axes[0].axhline(1, color='red', ls='--', lw=1)
axes[0].set_title('セクター平均 Beta vs Nikkei225')
axes[0].set_ylabel('Beta')
axes[0].set_xlabel('セクター')
axes[0].tick_params(axis='x', rotation=30)

# Right: individual betas
beta_df_sorted = beta_df.sort_values('beta', ascending=True)
colors_map = {'Manuf': 'tab:blue', 'Tech': 'tab:orange', 'Finance': 'tab:green',
              'Pharma': 'tab:red', 'Service': 'tab:purple', 'Industrial': 'tab:brown'}
bar_colors = [colors_map.get(s, 'gray') for s in beta_df_sorted['sector']]
axes[1].barh(beta_df_sorted['name'], beta_df_sorted['beta'], color=bar_colors, alpha=0.8)
axes[1].axvline(1, color='red', ls='--', lw=1)
axes[1].set_title('銘柄別 Beta vs Nikkei225')
axes[1].set_xlabel('Beta')

plt.tight_layout()
plt.show()

print('\nセクター別 Beta 統計:')
print(sector_beta)

### 6d. セクター内相関ヒートマップ

In [ ]:
pivot = df.dropna(subset=['logret_1d']).pivot_table(
    index='date', columns='symbol', values='logret_1d'
)

# Order columns by sector
sym_order = []
sector_labels = []
for sec in sector_order:
    syms_in_sec = [s for s, sv in SECTOR_MAP.items() if sv == sec and s in pivot.columns]
    sym_order.extend(syms_in_sec)
    sector_labels.extend([sec] * len(syms_in_sec))

pivot_ordered = pivot[sym_order]
corr_mat = pivot_ordered.corr()

# Rename for display
corr_mat.index   = [SYMBOL_NAME.get(s, s) for s in corr_mat.index]
corr_mat.columns = [SYMBOL_NAME.get(s, s) for s in corr_mat.columns]

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            annot_kws={'size': 7}, ax=ax)
ax.set_title('銘柄間リターン相関マトリクス (セクター順)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. 市場レジーム検出

### 7a–b. ローリング60日実現ボラティリティ & レジーム分類

In [ ]:
# Rolling 60d annualised vol of cross-sectional average return
rolling_vol = avg_ret.rolling(60).std() * np.sqrt(252)
rolling_vol.name = 'rolling_vol'

threshold_60p = rolling_vol.quantile(0.60)
regime_series = pd.Series(
    np.where(rolling_vol >= threshold_60p, 'High Vol', 'Low Vol'),
    index=rolling_vol.index,
    name='regime'
)
# Drop NaN windows
valid_idx = rolling_vol.dropna().index
regime_series = regime_series.loc[valid_idx]

print(f'60th-percentile threshold: {threshold_60p:.4f}')
print(regime_series.value_counts())

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(rolling_vol, color='navy', lw=1.2, label='60d ローリングVol (年率換算)')
ax.axhline(threshold_60p, color='red', ls='--', lw=1, label=f'60th pct ({threshold_60p:.3f})')
ax.fill_between(rolling_vol.index,
                rolling_vol.where(rolling_vol >= threshold_60p),
                threshold_60p, alpha=0.3, color='red', label='High Vol 領域')
ax.set_title('ローリング60日実現ボラティリティ (年率)', fontsize=13)
ax.set_ylabel('年率ボラティリティ')
ax.legend()
plt.tight_layout()
plt.show()

### 7c. レジーム別リターン KDE

In [ ]:
# Map regime to each row of df
df2 = df.dropna(subset=['logret_1d']).copy()
df2 = df2.join(regime_series.rename('regime'), on='date', how='left')
df2_valid = df2.dropna(subset=['regime'])

fig, ax = plt.subplots(figsize=(12, 5))
palette = {'High Vol': 'tomato', 'Low Vol': 'steelblue'}
for regime, grp in df2_valid.groupby('regime'):
    sns.kdeplot(grp['logret_1d'], ax=ax, label=regime,
                color=palette[regime], fill=True, alpha=0.35)

ax.set_title('レジーム別 日次対数リターン KDE', fontsize=13)
ax.set_xlabel('日次対数リターン')
ax.set_ylabel('密度')
ax.legend()
plt.tight_layout()
plt.show()

### 7d. レジーム別 Sharpe 比

In [ ]:
sharpe_rows = []
for regime, grp in df2_valid.groupby('regime'):
    r = grp['logret_1d']
    ann_ret = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
    sharpe_rows.append({'regime': regime, 'ann_return': ann_ret,
                        'ann_vol': ann_vol, 'sharpe': sharpe,
                        'n_obs': len(r)})

sharpe_df = pd.DataFrame(sharpe_rows)
print('レジーム別 Sharpe 比:')
sharpe_df.style.format({
    'ann_return': '{:.4f}', 'ann_vol': '{:.4f}', 'sharpe': '{:.3f}'
})

---
## 8. ドローダウン分析

### 8a. 銘柄別最大ドローダウン

In [ ]:
dd_store = {}
dd_rows  = []

for sym, grp in df.groupby('symbol'):
    g = grp[['date', 'logret_1d']].dropna().sort_values('date').set_index('date')
    cumret = g['logret_1d'].cumsum()
    running_max = cumret.cummax()
    drawdown = cumret - running_max
    dd_store[sym] = drawdown
    dd_rows.append({
        'symbol': sym,
        'name': SYMBOL_NAME.get(sym, sym),
        'sector': SECTOR_MAP.get(sym, 'Other'),
        'max_dd': drawdown.min(),
        'max_dd_date': drawdown.idxmin().date() if not drawdown.empty else None,
    })

dd_df = pd.DataFrame(dd_rows).sort_values('max_dd').reset_index(drop=True)
print('最大ドローダウン TOP10 (悪い順):')
dd_df.head(10).style.format({'max_dd': '{:.4f}'})

### 8b. ローリング252日最大ドローダウン (代表銘柄)

In [ ]:
# Pick top-5 by |max_dd| and a few names
highlight_syms = dd_df.head(5)['symbol'].tolist()

fig, ax = plt.subplots(figsize=(14, 5))
for sym in highlight_syms:
    dd = dd_store[sym]
    rolling_dd = dd.rolling(252).min()
    ax.plot(rolling_dd, label=SYMBOL_NAME.get(sym, sym), lw=1.2)

ax.set_title('ローリング252日最大ドローダウン (上位5銘柄)', fontsize=13)
ax.set_ylabel('ドローダウン (対数リターン累積)')
ax.legend()
plt.tight_layout()
plt.show()

### 8c. 回復期間ヒストグラム

In [ ]:
recovery_days = []

for sym, grp in df.groupby('symbol'):
    g = grp[['date', 'logret_1d']].dropna().sort_values('date').reset_index(drop=True)
    cumret = g['logret_1d'].cumsum()
    running_max = cumret.cummax()
    drawdown = cumret - running_max

    # Find troughs: local minima of drawdown series
    in_drawdown = False
    trough_val  = 0.0
    trough_idx  = None
    peak_val    = 0.0

    for i in range(len(drawdown)):
        dd_i  = drawdown.iloc[i]
        rm_i  = running_max.iloc[i]

        if dd_i < -0.01 and not in_drawdown:
            in_drawdown = True
            peak_val    = cumret.iloc[i - 1] if i > 0 else 0.0
            trough_val  = dd_i
            trough_idx  = i

        elif in_drawdown:
            if dd_i < trough_val:
                trough_val = dd_i
                trough_idx = i
            if cumret.iloc[i] >= peak_val:
                recovery_days.append(i - trough_idx)
                in_drawdown = False

recovery_days = [d for d in recovery_days if 0 < d < 500]

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(recovery_days, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
ax.axvline(np.median(recovery_days), color='red', ls='--', lw=1.5,
           label=f'中央値: {np.median(recovery_days):.0f}日')
ax.set_title('ドローダウン回復期間の分布 (全銘柄)', fontsize=13)
ax.set_xlabel('回復までの営業日数')
ax.set_ylabel('件数')
ax.legend()
plt.tight_layout()
plt.show()
print(f'回復イベント数: {len(recovery_days)}  |  中央値: {np.median(recovery_days):.0f}日  |  平均: {np.mean(recovery_days):.0f}日')

---
## 9. サマリー統計表

銘柄ごとの年率リターン・ボラティリティ・Sharpe・歪度・尖度・最大ドローダウンをまとめる。

In [ ]:
summary_rows = []
for sym, grp in df.groupby('symbol'):
    s = grp['logret_1d'].dropna()
    if len(s) < 20:
        continue
    ann_ret = s.mean() * 252
    ann_vol = s.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else np.nan
    sk      = skew(s)
    ku      = kurtosis(s)
    max_dd  = dd_store[sym].min() if sym in dd_store else np.nan
    summary_rows.append({
        'Symbol': sym,
        'Name': SYMBOL_NAME.get(sym, sym),
        'Sector': SECTOR_MAP.get(sym, 'Other'),
        'Ann.Return': ann_ret,
        'Ann.Vol': ann_vol,
        'Sharpe': sharpe,
        'Skewness': sk,
        'Kurtosis': ku,
        'MaxDrawdown': max_dd,
    })

summary = pd.DataFrame(summary_rows).sort_values('Sharpe', ascending=False).reset_index(drop=True)

print('=== 銘柄別サマリー統計 (Sharpe 降順) ===')
summary.style.format({
    'Ann.Return':  '{:.4f}',
    'Ann.Vol':     '{:.4f}',
    'Sharpe':      '{:.3f}',
    'Skewness':    '{:.3f}',
    'Kurtosis':    '{:.3f}',
    'MaxDrawdown': '{:.4f}',
}).background_gradient(subset=['Sharpe'], cmap='RdYlGn') \
  .background_gradient(subset=['MaxDrawdown'], cmap='RdYlGn_r')

In [ ]:
# Visual bar chart of Sharpe by name
fig, ax = plt.subplots(figsize=(14, 5))
colors = ['steelblue' if v >= 0 else 'tomato' for v in summary['Sharpe']]
ax.bar(summary['Name'], summary['Sharpe'], color=colors, alpha=0.85)
ax.axhline(0, color='black', lw=0.8)
ax.set_title('銘柄別 年率 Sharpe 比 (全期間)', fontsize=13)
ax.set_ylabel('Sharpe 比')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## まとめ

| 発見 | 内容 |
|------|------|
| **ファットテール** | Jarque-Bera 検定で全/大多数の銘柄が正規性を棄却。尖度が高く、大きなリターン事象が頻発 |
| **ボラティリティクラスタリング** | 二乗リターンの自己相関が有意に正 → GARCH 系モデルが有効 |
| **市場レジーム** | High/Low Vol レジームでリターン分布が明確に異なる → レジーム特徴量を予測モデルに組み込む価値あり |
| **セクター差異** | Finance / Industrial は Beta > 1、Pharma は低 Beta で防衛的特性 |
| **ドローダウン** | 回復期間中央値は数十営業日。イベント駆動型の下落からの回復が遅い銘柄も存在 |

**次ステップ**: 特徴量エンジニアリング（ローリング統計・レジーム指標）→ LightGBM/LSTM 予測モデル構築